# Extended Lab: Linear Regression with Scikit-Learn SGDRegressor

**Course context:** Optional Lab C1_W2_Lab05 (Machine Learning Specialization) – extended for practice, alternates, simulation, and responsible use.

## Goals
- Load multi-feature housing data (size, bedrooms, floors, age → price in $1000s).
- Apply z-score normalization with `StandardScaler`.
- Fit `sklearn.linear_model.SGDRegressor` (stochastic gradient descent).
- Inspect learned parameters, make predictions, evaluate, and visualize.
- Explore **alternate implementations**, extra practice drills, and a **Monte-Carlo simulation** of sensitivity to learning rate, sample size, and noise.
- Produce audience-adapted insights and understand model limitations.

## Cheat Sheet (keep this panel open while working)

| Task | Code / Concept |
|------|----------------|
| Load data | `np.loadtxt('data/houses.txt', delimiter=',')` → X = data[:, :4], y = data[:, 4] |
| Feature names | `['size(sqft)', 'bedrooms', 'floors', 'age']` |
| Normalize | `scaler = StandardScaler(); X_norm = scaler.fit_transform(X)` |
| Peak-to-peak | `np.ptp(X, axis=0)` – should shrink dramatically after scaling |
| Fit SGD | `sgdr = SGDRegressor(max_iter=1000, tol=1e-3); sgdr.fit(X_norm, y)` |
| Parameters | `w = sgdr.coef_`, `b = sgdr.intercept_` (on **normalized** scale) |
| Predict | `y_pred = sgdr.predict(X_norm)` or `X_norm @ w + b` |
| Metrics | `from sklearn.metrics import r2_score, mean_squared_error` |
| Inverse scale note | Coefficients are for z-scored features; interpret relative importance carefully |
| Common pitfall | Forgetting to scale → slow / failed convergence; scaling on full data before split (leakage) |
| Alternate OLS | `from sklearn.linear_model import LinearRegression` (closed-form, no iteration) |
| Pipeline | `make_pipeline(StandardScaler(), SGDRegressor(...))` |

---
## Flowchart of Desired Outcome

![Sklearn GD Flowchart](sklearn_gd_flowchart.png)

---

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=2, suppress=True)
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Libraries ready.")

## 1. Load the Dataset

The original lab uses `load_house_data()`. Here we load the same file directly.

**Columns:** size (sqft), bedrooms, floors, age (years), price ($1000s).

In [ ]:
data = np.loadtxt('data/houses.txt', delimiter=',')
X_train = data[:, :4]
y_train = data[:, 4]
X_features = ['size(sqft)', 'bedrooms', 'floors', 'age']

print(f"X shape: {X_train.shape}, y shape: {y_train.shape}")
print(f"First 3 rows of X:\n{X_train[:3]}")
print(f"First 3 prices (y): {y_train[:3]}")

### Quick EDA (added in extension)

In [ ]:
df = pd.DataFrame(X_train, columns=X_features)
df['price_1000s'] = y_train
print(df.describe().round(2))
print("\nCorrelation with price:")
print(df.corr()['price_1000s'].sort_values(ascending=False).round(3))

## 2. Scale / Normalize the Training Data

SGD is sensitive to feature scale. `StandardScaler` performs z-score normalization:

$$z = \frac{x - \mu}{\sigma}$$

After scaling, peak-to-peak ranges become comparable.

In [ ]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_train)

print(f"Peak to Peak range by column in Raw        X: {np.ptp(X_train, axis=0)}")
print(f"Peak to Peak range by column in Normalized X: {np.ptp(X_norm, axis=0)}")
print(f"\nScaler mean_: {scaler.mean_}")
print(f"Scaler scale_ (std): {scaler.scale_}")

## 3. Create and Fit the SGD Regression Model

In [ ]:
sgdr = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgdr.fit(X_norm, y_train)

print(sgdr)
print(f"number of iterations completed: {sgdr.n_iter_}, number of weight updates: {sgdr.t_}")

## 4. View Parameters

Parameters are associated with the *normalized* input data. They should be close to the values obtained in the pure-Python GD lab.

In [ ]:
b_norm = sgdr.intercept_
w_norm = sgdr.coef_
print(f"model parameters:                   w: {w_norm}, b: {b_norm}")
print( "model parameters from previous lab: w: [110.56 -21.27 -32.71 -37.97], b: 363.16")

## 5. Make Predictions & Evaluate

In [ ]:
# predict with the model
y_pred_sgd = sgdr.predict(X_norm)
# manual computation for verification
y_pred = np.dot(X_norm, w_norm) + b_norm
print(f"prediction using np.dot() and sgdr.predict match: {(np.allclose(y_pred, y_pred_sgd))}")

print(f"\nPrediction on training set (first 4):\n{y_pred[:4]}")
print(f"Target values (first 4):\n{y_train[:4]}")

r2 = r2_score(y_train, y_pred)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
mae = mean_absolute_error(y_train, y_pred)
print(f"\nR² = {r2:.4f} | RMSE = {rmse:.2f} ($1000s) | MAE = {mae:.2f} ($1000s)")

## 6. Plot Results – Target vs Prediction by Feature

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X_train[:, i], y_train, label='target', alpha=0.7, s=40)
    ax[i].scatter(X_train[:, i], y_pred, color='orange', label='predict', alpha=0.7, s=40)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel('Price ($1000s)')
ax[0].legend()
fig.suptitle('Target versus prediction using z-score normalized SGD model', fontsize=12)
plt.tight_layout()
plt.savefig('sklearn_gd_pred_vs_target.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Alternate Implementations

### 7.1 Closed-form OLS with `LinearRegression` (no iteration, exact for linear case)

In [ ]:
ols = LinearRegression()
ols.fit(X_norm, y_train)
print(f"OLS w: {ols.coef_}, b: {ols.intercept_}")
print(f"SGD  w: {w_norm}, b: {b_norm}")
print(f"R² OLS: {r2_score(y_train, ols.predict(X_norm)):.4f}")

### 7.2 Manual z-score + pure NumPy batch GD (educational)

In [ ]:
def manual_zscore(X):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    return (X - mu) / sigma, mu, sigma

X_man, mu, sigma = manual_zscore(X_train)

def compute_cost(X, y, w, b):
    m = X.shape[0]
    f = X @ w + b
    return np.sum((f - y)**2) / (2 * m)

def gradient_descent(X, y, w, b, alpha, num_iters):
    m = X.shape[0]
    J_hist = []
    for i in range(num_iters):
        f = X @ w + b
        err = f - y
        dw = (X.T @ err) / m
        db = np.sum(err) / m
        w = w - alpha * dw
        b = b - alpha * db
        if i % 100 == 0:
            J_hist.append(compute_cost(X, y, w, b))
    return w, b, J_hist

w_init = np.zeros(X_man.shape[1])
b_init = 0.0
w_gd, b_gd, J_hist = gradient_descent(X_man, y_train, w_init, b_init, alpha=0.1, num_iters=1000)
print(f"Manual GD w: {w_gd}, b: {b_gd}")
print(f"Final cost: {J_hist[-1]:.2f}")

### 7.3 Pipeline (best practice for production)

In [ ]:
pipe = make_pipeline(StandardScaler(), SGDRegressor(max_iter=1000, random_state=42))
pipe.fit(X_train, y_train)  # note: raw X, scaler inside
print(f"Pipeline R²: {pipe.score(X_train, y_train):.4f}")
print(f"Pipeline coef (normalized): {pipe.named_steps['sgdregressor'].coef_}")

---
## 8. More Practice

### Practice A – New house prediction
Predict price for a house of 1600 sqft, 3 bedrooms, 1 floor, 40 years old.

In [ ]:
new_house = np.array([[1600, 3, 1, 40]])
new_house_norm = scaler.transform(new_house)
pred_price = sgdr.predict(new_house_norm)[0]
print(f"Predicted price: ${pred_price * 1000:,.0f}")

### Practice B – Drop the weakest feature and re-fit
Which feature has the smallest |coefficient|? Remove it and compare R².

In [ ]:
print("Absolute normalized coefficients:", np.abs(w_norm))
# bedrooms often has smaller magnitude in this data
mask = [0, 2, 3]  # keep size, floors, age
X_red = X_train[:, mask]
scaler_red = StandardScaler()
X_red_norm = scaler_red.fit_transform(X_red)
sgdr_red = SGDRegressor(max_iter=1000, random_state=42).fit(X_red_norm, y_train)
print(f"Full model R²: {r2:.4f}")
print(f"Reduced model R²: {r2_score(y_train, sgdr_red.predict(X_red_norm)):.4f}")

### Practice C – Learning-rate / max_iter sensitivity (single run)

In [ ]:
results = []
for eta in [0.001, 0.01, 0.1]:
    for maxit in [100, 500, 2000]:
        model = SGDRegressor(max_iter=maxit, eta0=eta, learning_rate='constant',
                             tol=None, random_state=42)
        model.fit(X_norm, y_train)
        results.append({
            'eta0': eta, 'max_iter': maxit,
            'n_iter': model.n_iter_,
            'R2': r2_score(y_train, model.predict(X_norm))
        })
pd.DataFrame(results).sort_values('R2', ascending=False).head(8)

---
## 9. Simulation Section – Monte-Carlo Sensitivity

We generate synthetic data from a known linear model + noise, then vary:
- sample size `n`
- noise level `sigma`
- learning rate

and record R², recovered coefficients, and iterations.

In [ ]:
def simulate_once(n=100, sigma=20.0, eta0=0.01, max_iter=1000, true_w=None, true_b=300.0):
    if true_w is None:
        true_w = np.array([0.15, -15.0, -20.0, -1.5])  # approx un-normalized intuition
    # generate features roughly matching house ranges
    size = np.random.uniform(800, 3000, n)
    beds = np.random.randint(1, 6, n)
    floors = np.random.randint(1, 3, n)
    age = np.random.uniform(5, 90, n)
    X = np.column_stack([size, beds, floors, age])
    y = X @ true_w + true_b + np.random.normal(0, sigma, n)

    scaler = StandardScaler()
    Xn = scaler.fit_transform(X)
    model = SGDRegressor(max_iter=max_iter, eta0=eta0, learning_rate='constant',
                         tol=1e-4, random_state=None)
    model.fit(Xn, y)
    r2 = r2_score(y, model.predict(Xn))
    return {
        'n': n, 'sigma': sigma, 'eta0': eta0,
        'R2': r2, 'n_iter': model.n_iter_,
        'w_norm': model.coef_.copy(), 'b': model.intercept_[0]
    }

# Monte-Carlo over noise levels
np.random.seed(123)
noise_levels = [5, 15, 30, 50]
mc_results = []
for sig in noise_levels:
    for trial in range(30):
        mc_results.append(simulate_once(n=120, sigma=sig))

mc_df = pd.DataFrame(mc_results)
summary = mc_df.groupby('sigma')['R2'].agg(['mean', 'std', 'min', 'max'])
print(summary.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mc_df.boxplot(column='R2', by='sigma', ax=axes[0])
axes[0].set_title('R² vs Noise Level (σ)')
axes[0].set_xlabel('Noise σ')
axes[0].get_figure().suptitle('')

# sample-size effect
ns = [30, 60, 100, 200]
size_results = []
for n in ns:
    for _ in range(25):
        size_results.append(simulate_once(n=n, sigma=20))
size_df = pd.DataFrame(size_results)
size_df.boxplot(column='R2', by='n', ax=axes[1])
axes[1].set_title('R² vs Sample Size')
axes[1].set_xlabel('n')
axes[1].get_figure().suptitle('')

plt.tight_layout()
plt.savefig('sklearn_gd_simulation.png', dpi=120, bbox_inches='tight')
plt.show()

### Interactive-style parameter playground (change values and re-run)

In [ ]:
# === MODIFY THESE ===
SIM_N = 80
SIM_SIGMA = 25.0
SIM_ETA = 0.05
SIM_MAX_ITER = 800
# ====================

res = simulate_once(n=SIM_N, sigma=SIM_SIGMA, eta0=SIM_ETA, max_iter=SIM_MAX_ITER)
print(f"n={SIM_N}, σ={SIM_SIGMA}, η={SIM_ETA} → R²={res['R2']:.3f}, iterations={res['n_iter']}")
print(f"Recovered normalized w: {res['w_norm']}, b≈{res['b']:.1f}")

---
## 10. What the Model Can and Cannot Predict (Responsible Use Notes)

**Can:**
- Capture linear (or approximately linear) relationships between the chosen numeric features and price *within the range of the training data*.
- Provide fast, scalable training via SGD when data volume grows.
- Give relative feature importance on the *normalized* scale (larger |w| → stronger linear association after scaling).

**Cannot:**
- Model strong non-linearities or interactions unless you engineer those features yourself.
- Guarantee good predictions for houses far outside the observed size/age/bedroom ranges (extrapolation risk).
- Infer causality ("adding a bedroom will raise price by $X") – observational data only.
- Produce calibrated uncertainty intervals out of the box (use residual analysis, bootstrap, or Bayesian alternatives).
- Automatically handle missing values, categorical variables, or multicollinearity diagnostics.

**Responsible practices:**
1. Always report R² / RMSE *and* residual plots.
2. Document the training data distribution and warn about out-of-distribution inputs.
3. Prefer pipelines so scaling is never forgotten at inference time.
4. For high-stakes decisions (loan underwriting, tax assessment) combine with domain rules and human review.
5. Adapt language to the audience (see companion Strategy / Audience documents).

---
## Congratulations!
You have:
- used scikit-learn’s SGDRegressor with proper feature scaling,
- compared it to OLS and pure NumPy GD,
- practiced prediction and feature ablation,
- run Monte-Carlo simulations of noise and sample-size effects,
- and reflected on limitations and responsible reporting.

Next steps: try `SGDRegressor` with `penalty='l1'` or `'elasticnet'`, or wrap the whole thing in a cross-validation loop.